# IIC 2440 – Procesamiento de Datos Masivos
## Tarea 1 — Parte 2: Analítica y MapReduce

**Instrucciones de uso:**
- Ajusta `WAREHOUSE_PATH` y las rutas de dimensiones en la celda de configuración.
- Ejecuta las secciones en orden (00 → 06).
- No se usa PySpark ni Pandas: todo el procesamiento usa `map()`, `functools.reduce()` e iteradores Python.


---
## 00 — Imports y configuración

In [ ]:
import os
import sys
import math
import collections
import functools
import itertools
import re
import unicodedata
from datetime import datetime, timedelta

# Agrega la carpeta del proyecto al path para importar utils
sys.path.insert(0, os.path.dirname(os.path.abspath("utils_mapreduce.py")))
from utilidades import (
    leer_warehouse, leer_parquet_simple,
    cargar_dim_region, cargar_dim_source,
    tokenizar, normalizar, STOPWORDS,
    mapreduce, shuffle_and_sort, reduce_fn, map_fn,
    sumar, top_k
)

# ─── AJUSTA ESTAS RUTAS ───────────────────────────────────────────────────
WAREHOUSE_PATH  = "warehouse/fact_news"       # carpeta raíz de fact_news
DIM_REGION_PATH = "warehouse/dim_region.parquet"
DIM_SOURCE_PATH = "warehouse/dim_source.parquet"
DIM_DATE_PATH   = "warehouse/dim_date.parquet"
# ──────────────────────────────────────────────────────────────────────────

print("✓ Imports OK")


---
## 01 — Exploración rápida del warehouse

In [ ]:
# Cargar dimensiones (son pequeñas, caben en memoria)
dim_region = cargar_dim_region(DIM_REGION_PATH)
dim_source = cargar_dim_source(DIM_SOURCE_PATH)

print(f"Regiones disponibles ({len(dim_region)}):")
for sk, nombre in sorted(dim_region.items()):
    print(f"  {sk:3d} → {nombre}")

print(f"\nFuentes disponibles: {len(dim_source)}")


In [ ]:
# Conteo rápido de artículos por año-mes (sin cargar todo a memoria)
conteo_particiones = {}
for entry in sorted(os.listdir(WAREHOUSE_PATH)):
    if not entry.startswith("year="): continue
    anio = int(entry.split("=")[1])
    year_path = os.path.join(WAREHOUSE_PATH, entry)
    for sub in sorted(os.listdir(year_path)):
        if not sub.startswith("month="): continue
        mes = int(sub.split("=")[1])
        month_path = os.path.join(year_path, sub)
        n = sum(
            __import__("pyarrow.parquet", fromlist=["read_table"])
                .read_table(os.path.join(month_path, f)).num_rows
            for f in os.listdir(month_path) if f.endswith(".parquet")
        )
        conteo_particiones[f"{anio}-{mes:02d}"] = n

total = sum(conteo_particiones.values())
print(f"Total artículos en warehouse: {total:,}")
print("\nDistribución por año-mes:")
for ym, n in sorted(conteo_particiones.items()):
    barra = "█" * (n // 5000)
    print(f"  {ym}: {n:7,}  {barra}")


---
## 02 — Tarea 2.1: Top-K términos mensuales

**Objetivo:** Los 20 términos más frecuentes por año-mes.

**Diseño MapReduce:**
- **Map:** `artículo → [(año-mes, palabra, 1) for palabra in tokenizar(title + body)]`
  → emite pares `((año-mes, palabra), 1)`
- **Reduce:** suma por clave `(año-mes, palabra)`
- **Post-proceso:** top-20 por mes (resultado ya es pequeño, Python estándar)


In [ ]:
def mapper_topk(articulo):
    """
    Recibe un dict de fact_news.
    Emite pares ((año-mes, palabra), 1) para cada token del título y cuerpo.
    """
    año  = articulo.get("year")
    mes  = articulo.get("month")
    ym   = f"{año}-{mes:02d}"
    texto = (articulo.get("title") or "") + " " + (articulo.get("body") or "")
    tokens = tokenizar(texto)
    return [((ym, token), 1) for token in tokens]


# ── Pasada MapReduce ───────────────────────────────────────────────────────
print("Ejecutando MapReduce 2.1... (puede tardar unos minutos)")

# map: genera lista de listas de pares, la aplanamos
pares_raw = list(map_fn(mapper_topk, leer_warehouse(WAREHOUSE_PATH)))
pares_planos = list(itertools.chain.from_iterable(pares_raw))

# shuffle + reduce: suma por (año-mes, palabra)
grupos = shuffle_and_sort(pares_planos)
freq_ym_palabra = dict(reduce_fn(sumar, grupos))

print(f"Pares únicos (año-mes, palabra): {len(freq_ym_palabra):,}")


In [ ]:
# Post-proceso: top-20 por mes
# Reagrupar por año-mes
freq_por_mes = collections.defaultdict(list)
for (ym, palabra), cnt in freq_ym_palabra.items():
    freq_por_mes[ym].append((palabra, cnt))

# Top-20 por mes
topk_mensual = {
    ym: sorted(pares, key=lambda x: x[1], reverse=True)[:20]
    for ym, pares in freq_por_mes.items()
}

# Mostrar resultado
print("Formato: año-mes, término, frecuencia\n")
for ym in sorted(topk_mensual.keys()):
    print(f"── {ym} ──")
    for palabra, cnt in topk_mensual[ym][:5]:   # mostramos los 5 primeros
        print(f"   {ym}, {palabra}, {cnt}")
    print()


In [ ]:
# Guardar resultado completo en CSV
import csv

os.makedirs("resultados", exist_ok=True)
with open("resultados/2_1_topk_mensual.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["anio_mes", "termino", "frecuencia"])
    for ym in sorted(topk_mensual.keys()):
        for palabra, cnt in topk_mensual[ym]:
            writer.writerow([ym, palabra, cnt])

print("✓ Guardado en resultados/2_1_topk_mensual.csv")


---
## 03 — Tarea 2.2: Distribución de palabras por región

**Objetivo:** Para cada par `(región, término)`: ocurrencias regionales,
ocurrencias globales y razón `regional / global`.

**Diseño MapReduce (2 pasadas):**
- **Pasada 1 — frecuencias globales:** Map emite `(palabra, 1)` → Reduce suma → filtra `freq >= K`
- **Pasada 2 — frecuencias regionales:** Map emite `((región, palabra), 1)` → Reduce suma
- **Post-proceso:** join de ambos resultados + cálculo de razón


In [ ]:
K_FRECUENCIA_MINIMA = 50   # Solo palabras con freq global >= K

# ── Pasada 1: frecuencias globales ───────────────────────────────────────
print("Pasada 1: frecuencias globales...")

def mapper_global(articulo):
    """Emite (palabra, 1) para cada token del artículo."""
    texto = (articulo.get("title") or "") + " " + (articulo.get("body") or "")
    return [(token, 1) for token in tokenizar(texto)]

pares_global = list(itertools.chain.from_iterable(
    map_fn(mapper_global, leer_warehouse(WAREHOUSE_PATH))
))
grupos_global = shuffle_and_sort(pares_global)
freq_global = dict(reduce_fn(sumar, grupos_global))

# Filtrar por frecuencia mínima
vocab_filtrado = {p: c for p, c in freq_global.items() if c >= K_FRECUENCIA_MINIMA}
print(f"Vocabulario con freq >= {K_FRECUENCIA_MINIMA}: {len(vocab_filtrado):,} palabras")


In [ ]:
# ── Pasada 2: frecuencias regionales ─────────────────────────────────────
print("Pasada 2: frecuencias regionales...")

# Excluir región "Desconocida" (region_sk de is_known_region=False)
dim_reg_completo = leer_parquet_simple(DIM_REGION_PATH)
regiones_conocidas = {
    f["region_sk"] for f in dim_reg_completo if f["is_known_region"]
}

def mapper_regional(articulo):
    """
    Emite ((region_sk, palabra), 1) solo para artículos con región conocida
    y palabras que están en el vocabulario filtrado.
    """
    region_sk = articulo.get("region_sk")
    if region_sk not in regiones_conocidas:
        return []
    texto = (articulo.get("title") or "") + " " + (articulo.get("body") or "")
    return [
        ((region_sk, token), 1)
        for token in tokenizar(texto)
        if token in vocab_filtrado
    ]

pares_reg = list(itertools.chain.from_iterable(
    map_fn(mapper_regional, leer_warehouse(WAREHOUSE_PATH))
))
grupos_reg = shuffle_and_sort(pares_reg)
freq_regional = dict(reduce_fn(sumar, grupos_reg))

print(f"Pares únicos (región, palabra): {len(freq_regional):,}")


In [ ]:
# ── Post-proceso: calcular razón regional/global ─────────────────────────
resultados_22 = []
for (region_sk, palabra), freq_reg in freq_regional.items():
    freq_glob = vocab_filtrado.get(palabra, 1)
    razon = freq_reg / freq_glob
    resultados_22.append({
        "region_sk":   region_sk,
        "region":      dim_region.get(region_sk, "?"),
        "palabra":      palabra,
        "freq_regional": freq_reg,
        "freq_global":   freq_glob,
        "razon":         round(razon, 6),
    })

# Ordenar por razón desc para ver palabras distintivas por región
resultados_22.sort(key=lambda x: x["razon"], reverse=True)

print("Top 5 palabras con mayor concentración regional:")
for r in resultados_22[:15]:
    print(f"  {r['region']:25s} | {r['palabra']:20s} | reg={r['freq_regional']:6d} | glob={r['freq_global']:8d} | razón={r['razon']:.4f}")


In [ ]:
# Guardar resultado
with open("resultados/2_2_distribucion_regional.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["region_sk","region","palabra","freq_regional","freq_global","razon"])
    writer.writeheader()
    writer.writerows(resultados_22)
print("✓ Guardado en resultados/2_2_distribucion_regional.csv")


---
## 04 — Tarea 2.3: Divergencia de vocabulario por fuente

**Objetivo:** Para cada medio, calcular qué tan distinto es su vocabulario
respecto a la distribución global (métrica tipo KL aproximada).

**Diseño MapReduce:**
- **Map:** `artículo → ((source_sk, palabra), 1)`
- **Reduce:** suma por `(source_sk, palabra)` → distribución por fuente
- **Post-proceso:** KL divergence aproximada `sum(p * log(p/q))` para cada fuente
  - `p` = distribución de frecuencias relativas de la fuente
  - `q` = distribución global (de la Pasada 1 de 2.2)


In [ ]:
print("MapReduce 2.3: frecuencias por fuente...")

def mapper_fuente(articulo):
    """Emite ((source_sk, palabra), 1) para cada token."""
    source_sk = articulo.get("source_sk")
    texto = (articulo.get("title") or "") + " " + (articulo.get("body") or "")
    return [
        ((source_sk, token), 1)
        for token in tokenizar(texto)
        if token in vocab_filtrado   # reutilizamos vocabulario filtrado de 2.2
    ]

pares_fuente = list(itertools.chain.from_iterable(
    map_fn(mapper_fuente, leer_warehouse(WAREHOUSE_PATH))
))
grupos_fuente = shuffle_and_sort(pares_fuente)
freq_por_fuente = dict(reduce_fn(sumar, grupos_fuente))
print(f"Pares únicos (fuente, palabra): {len(freq_por_fuente):,}")


In [ ]:
# Calcular distribución global normalizada (q)
total_global = sum(vocab_filtrado.values())
q_global = {p: c / total_global for p, c in vocab_filtrado.items()}

# Agrupar frecuencias por fuente
fuente_vocab = collections.defaultdict(dict)
for (source_sk, palabra), cnt in freq_por_fuente.items():
    fuente_vocab[source_sk][palabra] = cnt

# Calcular KL aproximada para cada fuente
# KL(P||Q) = sum_x P(x) * log(P(x) / Q(x))  — usando solo palabras en vocab
kl_divergencias = []
for source_sk, vocab in fuente_vocab.items():
    total_fuente = sum(vocab.values())
    if total_fuente < 500:   # Ignorar fuentes con muy pocos artículos
        continue
    kl = 0.0
    for palabra, cnt in vocab.items():
        p_x = cnt / total_fuente
        q_x = q_global.get(palabra, 1e-10)
        kl += p_x * math.log(p_x / q_x)
    kl_divergencias.append({
        "source_sk":   source_sk,
        "fuente":      dim_source.get(source_sk, "?"),
        "kl_divergencia": round(kl, 6),
        "total_palabras": total_fuente,
    })

kl_divergencias.sort(key=lambda x: x["kl_divergencia"], reverse=True)

print(f"Fuentes analizadas: {len(kl_divergencias)}")
print("\nTop 10 fuentes con mayor divergencia (vocabulario más distinto al global):")
for r in kl_divergencias[:10]:
    print(f"  {r['fuente']:30s} KL={r['kl_divergencia']:.4f}  (palabras={r['total_palabras']:,})")


In [ ]:
with open("resultados/2_3_divergencia_fuentes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["source_sk","fuente","kl_divergencia","total_palabras"])
    writer.writeheader()
    writer.writerows(kl_divergencias)
print("✓ Guardado en resultados/2_3_divergencia_fuentes.csv")


---
## 05 — Tarea 2.4: Detección de peaks

**Objetivo:** Detectar días con volumen anormalmente alto de artículos.

**Diseño:**
- **Pasada MapReduce:** Map emite `(fecha, 1)` → Reduce suma → conteo diario
- **Post-proceso Python:** calcular promedio móvil de 7 días + detectar días
  donde el volumen supera `promedio_movil * umbral`


In [ ]:
print("MapReduce 2.4: conteo diario...")

def mapper_diario(articulo):
    """Emite (fecha_str, 1) para cada artículo."""
    fecha = articulo.get("publish_date")
    if hasattr(fecha, "strftime"):
        fecha = fecha.strftime("%Y-%m-%d")
    else:
        fecha = str(fecha)[:10]
    return (fecha, 1)

# Una sola pasada MapReduce
pares_diario = list(map_fn(mapper_diario, leer_warehouse(WAREHOUSE_PATH)))
grupos_diario = shuffle_and_sort(pares_diario)
conteo_diario = dict(reduce_fn(sumar, grupos_diario))

print(f"Días con actividad: {len(conteo_diario)}")
print(f"Rango: {min(conteo_diario)} → {max(conteo_diario)}")


In [ ]:
# Post-proceso: promedio móvil y detección de peaks
# (resultado diario es pequeño, Python estándar)

VENTANA_DIAS   = 7
UMBRAL_FACTOR  = 2.0   # Peak si volumen > umbral * promedio_movil

fechas_ordenadas = sorted(conteo_diario.keys())
volumenes = [conteo_diario[f] for f in fechas_ordenadas]

def promedio_movil(serie, ventana):
    """Calcula promedio móvil centrado en i usando ventana días anteriores."""
    resultado = []
    for i in range(len(serie)):
        inicio = max(0, i - ventana)
        ventana_vals = serie[inicio:i]
        if not ventana_vals:
            resultado.append(serie[i])
        else:
            resultado.append(sum(ventana_vals) / len(ventana_vals))
    return resultado

promedios = promedio_movil(volumenes, VENTANA_DIAS)

peaks = []
for i, (fecha, vol) in enumerate(zip(fechas_ordenadas, volumenes)):
    prom = promedios[i]
    if prom > 0 and vol > UMBRAL_FACTOR * prom:
        peaks.append({
            "fecha":           fecha,
            "volumen":         vol,
            "promedio_movil":  round(prom, 1),
            "factor":          round(vol / prom, 2),
        })

peaks.sort(key=lambda x: x["factor"], reverse=True)

print(f"Peaks detectados (factor > {UMBRAL_FACTOR}): {len(peaks)}")
print("\nTop 15 peaks:")
print(f"{'Fecha':12s}  {'Volumen':>8s}  {'Promedio':>10s}  {'Factor':>7s}")
print("-" * 45)
for p in peaks[:15]:
    print(f"{p['fecha']:12s}  {p['volumen']:8,d}  {p['promedio_movil']:10.1f}  {p['factor']:7.2f}x")


In [ ]:
with open("resultados/2_4_peaks.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["fecha","volumen","promedio_movil","factor"])
    writer.writeheader()
    writer.writerows(peaks)
print("✓ Guardado en resultados/2_4_peaks.csv")


---
## 06 — Tarea 2.5: Visualizaciones e informe

Generamos visualizaciones que apoyen el análisis de:
- Tendencias en cobertura mediática
- Vocabulario distintivo por región
- Divergencia por fuente
- Peaks detectados


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

os.makedirs("resultados/figuras", exist_ok=True)

# Estilo limpio
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})


In [ ]:
# ── Figura 1: Volumen diario + peaks ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

fechas_dt = [datetime.strptime(f, "%Y-%m-%d") for f in fechas_ordenadas]

ax.fill_between(fechas_dt, volumenes, alpha=0.25, color="#4472C4")
ax.plot(fechas_dt, volumenes, lw=0.8, color="#4472C4", label="Volumen diario")
ax.plot(fechas_dt, promedios, lw=1.5, color="#ED7D31", ls="--", label=f"Promedio móvil {VENTANA_DIAS}d")

# Marcar peaks
peaks_fechas = [datetime.strptime(p["fecha"], "%Y-%m-%d") for p in peaks[:10]]
peaks_vols   = [conteo_diario[p["fecha"]] for p in peaks[:10]]
ax.scatter(peaks_fechas, peaks_vols, color="#FF0000", zorder=5, s=40, label="Peak detectado")

ax.set_title("Volumen diario de artículos — noticias chilenas 2023-2025")
ax.set_ylabel("Artículos / día")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig("resultados/figuras/fig1_volumen_peaks.png", bbox_inches="tight")
plt.show()
print("✓ fig1_volumen_peaks.png guardada")


In [ ]:
# ── Figura 2: Top-10 términos del mes más activo ──────────────────────────
mes_max = max(conteo_particiones, key=conteo_particiones.get)
top_mes = topk_mensual.get(mes_max, [])[:10]

if top_mes:
    palabras_bar = [t[0] for t in reversed(top_mes)]
    freqs_bar    = [t[1] for t in reversed(top_mes)]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(palabras_bar, freqs_bar, color="#4472C4", alpha=0.85)
    ax.bar_label(bars, fmt="%,d", padding=4, fontsize=9)
    ax.set_title(f"Top-10 términos más frecuentes — {mes_max}")
    ax.set_xlabel("Frecuencia")
    plt.tight_layout()
    plt.savefig("resultados/figuras/fig2_topk_mes.png", bbox_inches="tight")
    plt.show()
    print(f"✓ fig2_topk_mes.png guardada ({mes_max})")


In [ ]:
# ── Figura 3: Palabras más distintivas por región (top razón) ─────────────
# Seleccionar top-3 palabras por región según razón regional/global
from collections import defaultdict

top_por_region = defaultdict(list)
for r in resultados_22:
    top_por_region[r["region"]].append((r["palabra"], r["razon"]))

# Tomar las 5 regiones con más artículos y sus top-3 palabras
regiones_plot = sorted(
    top_por_region.keys(),
    key=lambda rg: sum(rr["freq_regional"] for rr in resultados_22 if rr["region"] == rg),
    reverse=True
)[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for ax, region in zip(axes, regiones_plot):
    datos = sorted(top_por_region[region], key=lambda x: x[1], reverse=True)[:8]
    if not datos: continue
    pals = [d[0] for d in reversed(datos)]
    razones = [d[1] for d in reversed(datos)]
    bars = ax.barh(pals, razones, color="#70AD47", alpha=0.85)
    ax.set_title(region, fontsize=10, fontweight="bold")
    ax.set_xlabel("Razón regional/global", fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle("Palabras más distintivas por región", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("resultados/figuras/fig3_palabras_region.png", bbox_inches="tight")
plt.show()
print("✓ fig3_palabras_region.png guardada")


In [ ]:
# ── Figura 4: Top-15 fuentes por divergencia KL ───────────────────────────
top_kl = kl_divergencias[:15]
fuentes_plot = [r["fuente"] for r in reversed(top_kl)]
kl_vals      = [r["kl_divergencia"] for r in reversed(top_kl)]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(fuentes_plot, kl_vals, color="#ED7D31", alpha=0.85)
ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=8)
ax.set_title("Top 15 medios con mayor divergencia de vocabulario (KL aprox.)")
ax.set_xlabel("KL Divergencia")
plt.tight_layout()
plt.savefig("resultados/figuras/fig4_kl_fuentes.png", bbox_inches="tight")
plt.show()
print("✓ fig4_kl_fuentes.png guardada")


---
## Análisis y conclusiones

> **Completa esta sección en el informe.**

### Tendencias generales
- (Describe los términos que dominaron cada período)
- (Menciona cambios en la cobertura entre 2023 y 2025)

### Vocabulario por región
- (¿Qué regiones tienen un vocabulario más diferenciado?)
- (¿Hay eventos locales que expliquen las palabras distintivas?)

### Divergencia por fuente
- (¿Qué tipo de medios tienen mayor divergencia? ¿Especializados, locales, nacionales?)
- (¿Hay algún patrón en los medios con vocabulario más genérico?)

### Peaks detectados
- (Analiza los 3–5 peaks más prominentes)
- (¿Qué eventos nacionales o internacionales podrían explicarlos?)
